# Hillsboro Growth & Development GIS Analysis

## HIL-005 — Buildings

Initial exploration of the City of Hillsboro Buildings GIS REST service.

**Source:** City of Hillsboro GIS  
**Layer:** Planning_BaseData / MapServer / 91  
**Date accessed:** August 25, 2026

### Purpose

Explore the structure, metadata, data quality, and analytical potential of the Hillsboro building footprint dataset before acquiring the complete dataset.

In [ ]:
import requests

url = "https://gis.hillsboro-oregon.gov/public/rest/services/public/Planning_BaseData/MapServer/91"

response = requests.get(
    url,
    params={"f": "json"}
)

data = response.json()

print(data["name"])
print(data["geometryType"])

In [ ]:
for field in data["fields"]:
    print(field["name"], "—", field["type"])

In [ ]:
count_url = url + "/query"

params = {
    "where": "1=1",
    "returnCountOnly": "true",
    "f": "json"
}

count_response = requests.get(count_url, params=params)
count_data = count_response.json()

print("Number of records:", count_data["count"])

In [ ]:
params = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "false",
    "resultRecordCount": 5,
    "f": "json"
}

response = requests.get(count_url, params=params)
sample = response.json()

for feature in sample["features"]:
    print(feature["attributes"])

In [ ]:
import pandas as pd

records = [feature["attributes"] for feature in sample["features"]]

df = pd.DataFrame(records)

df

In [ ]:
params = {
    "where": "YEAR_BUILT = 0",
    "returnCountOnly": "true",
    "f": "json"
}

response = requests.get(count_url, params=params)
result = response.json()

print("Buildings with YEAR_BUILT = 0:", result["count"])

In [ ]:
for code, meaning in [("0", "Active"), ("1", "Demoed"), ("2", "Permitted")]:

    params = {
        "where": f"STATUS = '{code}'",
        "returnCountOnly": "true",
        "f": "json"
    }

    response = requests.get(count_url, params=params)
    count = response.json()["count"]

    print(f"{meaning}: {count:,}")

In [ ]:
params = {
    "where": "YEAR_DEMOLISHED IS NOT NULL",
    "returnCountOnly": "true",
    "f": "json"
}

response = requests.get(count_url, params=params)
demolished_year_count = response.json()["count"]

print("Buildings with a demolition year:", demolished_year_count)

## Initial Findings

- The layer contains 43,686 building records.
- Geometry type is polygon.
- The current dataset contains no Demoed or Permitted records.
- `YEAR_BUILT = 0` occurs in 8,732 records (approximately 20% of the dataset) and should be treated as a potential missing/unknown value rather than a literal construction year.
- `YEAR_DEMOLISHED` is not populated in the current dataset.
- The `STATUS` field uses a coded domain: 0 = Active, 1 = Demoed, 2 = Permitted.
- The current dataset therefore appears most useful for analyzing the existing building stock rather than historical demolition activity.